# RGB 通道，特征的概念

In [ ]:
# 导入必要的库
from PIL import Image  # 用于图像处理的PIL库
import numpy as np  # 用于数值计算的NumPy库
import matplotlib.pyplot as plt  # 用于显示图像的matplotlib.pyplot库

In [ ]:
PIC = './cat.jpg'  # './cat.jpg'是图像文件的路径

In [ ]:
# 本代码展示如何使用 PIL, NumPy 和 matplotlib 处理并展示彩色 RGB 图像的不同颜色通道

# 打开彩色图像文件
image = Image.open(PIC) 

# 将PIL图像对象转换为NumPy数组
image_array = np.array(image)

# 图像的维度为 [high, width, channel]，[:,:,0] 代表是所有长宽以及通道0，也就是红色通道

# 分离RGB颜色通道中的红色通道
red_channel = image_array[:, :, 0]
# 分离RGB颜色通道中的绿色通道
green_channel = image_array[:, :, 1]
# 分离RGB颜色通道中的蓝色通道
blue_channel = image_array[:, :, 2]

# 使用matplotlib展示原始图像
plt.subplot(221), plt.imshow(image), plt.title("Original Image")
# 使用matplotlib展示蓝色通道图像，使用蓝色调的颜色映射
plt.subplot(222), plt.imshow(blue_channel, cmap="Blues"), plt.title("Blue Channel")
# 使用matplotlib展示绿色通道图像，使用绿色调的颜色映射
plt.subplot(223), plt.imshow(green_channel, cmap="Greens"), plt.title("Green Channel")
# 使用matplotlib展示红色通道图像，使用红色调的颜色映射
plt.subplot(224), plt.imshow(red_channel, cmap="Reds"), plt.title("Red Channel")
# 显示所有子图
plt.show()

# YUV 色彩空间

In [ ]:
# 这段代码用于演示如何将RGB图像转换为YUV格式，并展示其Y、U、V各通道的效果

import cv2  # 导入OpenCV库，用于图像处理
import matplotlib.pyplot as plt  # 导入matplotlib.pyplot，用于图像显示

# 使用OpenCV的imread函数读取指定路径的图像
rgb_image = cv2.imread(PIC)  # './cat.png'是图像的路径，需要根据实际情况修改

# 将读取的RGB图像转换为YUV格式
# cv2.cvtColor函数用于转换图像的颜色空间，这里从BGR转换为YUV
# OpenCV中默认读取的格式为BGR，而不是RGB
yuv_image = cv2.cvtColor(rgb_image, cv2.COLOR_BGR2YUV)

# 使用cv2.split函数分离YUV图像的三个通道（Y, U, V）
y_channel, u_channel, v_channel = cv2.split(yuv_image)

# 使用matplotlib的subplot函数和imshow函数显示四个子图
# 第一个子图为原始的RGB图像
plt.subplot(221), plt.imshow(cv2.cvtColor(rgb_image, cv2.COLOR_BGR2RGB)), plt.title(
    "Original RGB feature"
)
# 第二个子图为Y通道的图像，使用灰度图显示
plt.subplot(222), plt.imshow(y_channel, cmap="gray"), plt.title("Y channel")
# 第三个子图为U通道的图像，同样使用灰度图显示
plt.subplot(223), plt.imshow(u_channel, cmap="gray"), plt.title("U channel")
# 第四个子图为V通道的图像
plt.subplot(224), plt.imshow(v_channel, cmap="gray"), plt.title("V channel")
# 显示所有子图
plt.show()

# 灰度图

## 手搓一段灰度图转换 vs OPENCV 灰度图转换

In [ ]:

def rgb_to_grayscale_manual(rgb_array):
    """
    手动实现RGB到灰度图的转换
    使用标准加权系数：0.299 * R + 0.587 * G + 0.114 * B
    """
    # 提取RGB通道
    r = rgb_array[:, :, 0]
    g = rgb_array[:, :, 1]
    b = rgb_array[:, :, 2]
    
    # 应用加权公式
    gray = 0.299 * r + 0.587 * g + 0.114 * b
    
    # 确保值在0-255范围内并转换为uint8
    gray = np.clip(gray, 0, 255).astype(np.uint8)
    
    return gray


In [ ]:
# 打开彩色图像
color_image = Image.open(PIC)
# 自定义函数转换为灰度图
rgb_array = np.array(color_image) # 图像转成np.array
gray_array = rgb_to_grayscale_manual(rgb_array)
# OPENCV转换为灰度图, L 代表的是灰度图
gray_image = color_image.convert("L")
# 保存灰度图
gray_image.save("./gray_cat.jpg")

# 打印图片格式
print("彩色图片格式: " + color_image.mode)
print("灰度图片格式: " + gray_image.mode)

In [ ]:
# 显示结果
plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1)
plt.imshow(rgb_array)
plt.title('Original RGB Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(gray_array, cmap='gray')
plt.title('Grayscale Image (Manual)')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(gray_image, cmap='gray')
plt.title('Grayscale Image (OpenCV)')
plt.axis('off')
plt.show()


# 灰度图核心原理
灰度图是单通道图像，每个像素只有一个亮度值。转换的关键是如何将三个通道（R, G, B）的颜色信息合理映射为一个亮度值。符合人眼感知的公式是加权平均法（光度法）：
标准加权系数：$0.299 * R + 0.587 * G + 0.114 * B$

- 人眼敏感性：人眼对绿色最敏感，红色次之，蓝色最不敏感。因此，权重系数中 G > R > B。
- 亮度信息：灰度图反映的是图像的亮度（Luminance）信息，而亮度是RGB通道共同作用的结果，不是三个通道亮度的简单算术平均。

## 灰度图也可以反向转成RGB图吗？因为权重已知。
这是一个非常经典且重要的问题。答案是：不能真正地、完整地反向转换。

你的思路——“权重已知”——在逻辑上是正确的，但关键在于转换过程丢失了信息。

我们可以用一个比喻来理解：

- RGB → 灰度：就像你把一杯红果汁、一杯绿果汁和一杯蓝果汁，按照一个固定配方（0.299, 0.587, 0.114）混合成一杯新的灰色果汁。然后，你把原来的三杯果汁倒掉了。

- 灰度 → RGB（试图反向）：现在，你只有这杯混合后的灰色果汁。即使你知道配方，你也无法确定原来那三杯红色、绿色、蓝色果汁各自具体是多少。

核心：数学上的“多对一”映射--灰度化是一个多对一的函数：

**无数种不同的 (R, G, B) 组合，通过加权平均可以计算出同一个灰度值。**

## 从信息论角度看：信息熵的永久丢失
RGB图像（24位）：每个像素携带了 3个自由度（R, G, B三个独立变量）的信息。

灰度图（8位）：每个像素只剩下 1个自由度（亮度值）。
在转换过程中，另外 2个自由度（色彩信息）被永久丢弃并无法从结果中恢复。权重公式是丢弃信息的“方式”，而不是存储信息的“钥匙”。